# 5주차 자습 실습. 조건을 주는 VAE(CVAE)와 코드북을 쓰는 VAE(VQ-VAE)
*Week 5 self-study lab. A VAE with a condition (CVAE) and a VAE with a codebook (VQ-VAE)*

**제출하지 않습니다. 채점하지 않습니다.** 4주차 노트북 `W04_training_a_vae.ipynb` 를 먼저 끝낸 뒤 여세요.
*Nothing to submit, nothing graded. Finish the Week 4 notebook `W04_training_a_vae.ipynb` first.*

**이 실습의 목표 한 문장.** 라벨을 두 네트워크에 모두 넣으면 생성할 숫자를 고를 수 있게 된다는 것, 그리고 "가장 가까운 코드 고르기"가 막은 기울기를 한 줄로 통과시킨다는 것을 직접 확인합니다.
*Goal in one sentence. See for yourself that giving the label to both networks lets you choose the digit, and that one line lets the gradient pass the "pick the nearest code" step that blocks it.*

| 절 / Section | 하는 일 / What you do | 시간 / Time |
|---|---|---|
| §0 | 준비. MNIST 내려받기 / Setup, download MNIST | 2분 / 2 min |
| §1 | 예측 세 개를 먼저 적기 / Write three predictions first | 5분 / 5 min |
| §2–§4 | CVAE. 빈칸 1개, 학습, 숫자 골라 생성 / CVAE. One blank, train, generate a chosen digit | 20분 / 20 min |
| §5–§6 | 가장 가까운 코드와 straight-through. 빈칸 2개, 손계산과 대조 / Nearest code and straight-through. Two blanks, checked against a hand calculation | 15분 / 15 min |
| §7–§8 | VQ-VAE 학습, 코드북 사용량, 무작위 인덱스 / Train a VQ-VAE, codebook usage, random indices | 20분 / 20 min |
| §9 | 예측 다시 읽기 / Re-read your predictions | 5분 / 5 min |

슬라이드와 짝(4·5주차 본 덱): §2 「How Do We Tell the Network “This Is Class 3”?」 · 「CVAE Gives the Label to Both Networks」, §3 「Both Networks See y. The Prior Stays Fixed」, §4 「Choose the Label, Then Draw」, §5 「From Encoder Output to Quantized Latent」, §6 「Picking a Code Blocks the Gradient」 · 「The Straight-Through Fix (Encoder Learning)」, §7 「An Image Becomes a Grid of Indices」 · 「How Does the Codebook Learn?」 · 「The Commitment Term Moves the Encoder Instead」 · 「The Full VQ-VAE Loss」 · 「Why Is There No KL Loss in VQ-VAE?」, §8 「Why Do Random Code Grids Not Look Like Digits?」 · 「New Samples Need a Prior over Code Grids」.
*Matching slides in the Weeks 4 and 5 main deck: the same titles, section by section.*

**GPU 가 없어도 됩니다.** CPU 에서 전체 약 10~15분 학습합니다. Colab 메뉴 「런타임 → 런타임 유형 변경」에서 GPU 를 고르면 더 빠릅니다.
*No GPU needed. Training takes about 10 to 15 minutes on a CPU in total. In Colab, Runtime → Change runtime type → GPU makes it faster.*

## §0 준비 / Setup
아래 셀을 실행하면 라이브러리를 불러오고, 시드를 고정하고, MNIST 를 내려받습니다(약 30초).
*Run the cell below. It imports the libraries, fixes the seed and downloads MNIST (about 30 seconds).*

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(0)                      # 시드 고정 / fix the seed
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"   # GPU 가 있으면 GPU / GPU if available
print("장치 / device:", device)

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell just below and continue.\n"
)

# MNIST 손글씨 숫자. 이미지 60,000장, 각 28 x 28 픽셀, 값은 0 과 1 사이
# MNIST handwritten digits. 60,000 images, 28 x 28 pixels each, values between 0 and 1
train_set = datasets.MNIST("data", train=True, download=True, transform=transforms.ToTensor())
test_set  = datasets.MNIST("data", train=False, download=True, transform=transforms.ToTensor())
print("학습 이미지 / training images:", len(train_set), "  시험 이미지 / test images:", len(test_set))
# 기대값 / Expected   학습 60000, 시험 10000 / training 60000, test 10000

## §1 먼저 예측하기 / Predict first
코드를 돌리기 **전에** 아래 세 질문의 답을 한 줄씩 적어 두세요. 틀려도 괜찮습니다. §9 에서 다시 읽습니다.
*Before running anything, write a one-line answer to each question. Being wrong is fine. You will re-read them in §9.*

1. CVAE 에서 **같은 z** 에 라벨 y 만 0 부터 9 까지 바꿔 디코딩하면, 열 장의 그림에서 무엇이 바뀌고 무엇이 남을까요?
   *In a CVAE, decode the **same z** with the label y set to 0, 1, …, 9. Across the ten pictures, what changes and what stays?*
2. 라벨을 **인코더에만** 주고 디코더에는 주지 않으면, 생성할 때 숫자를 고를 수 있을까요?
   *If only the encoder gets the label and the decoder does not, can you choose the digit when generating?*
3. VQ-VAE 에서 코드북 크기가 K = 32 이면, 격자의 한 칸이 가질 수 있는 코드 벡터는 몇 가지일까요? 학습이 끝났을 때 32개가 모두 쓰일까요?
   *In a VQ-VAE with a codebook of K = 32, how many different code vectors can one grid cell take? After training, will all 32 be used?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your predictions here. This cell does not need to be run.
my_prediction_1 = ""
my_prediction_2 = ""
my_prediction_3 = ""

## §2 CVAE. 라벨을 두 네트워크에 모두 준다 / CVAE. Give the label to both networks

**라벨을 숫자 벡터로 바꾸기.** 라벨 y 는 0 에서 9 사이의 정수입니다. 네트워크에 정수 3 을 그대로 넣으면 "3 은 1 보다 크다" 같은, 뜻이 없는 크기 관계가 생깁니다. 그래서 길이 10 의 벡터에서 y 번째 자리만 1 로 둡니다. 이것을 **원-핫(one-hot) 벡터**라고 합니다.
*Turning the label into numbers. The label y is an integer from 0 to 9. Feeding the integer 3 directly would suggest a meaningless order such as "3 is bigger than 1". Instead we use a length-10 vector with a 1 only in position y. This is a **one-hot vector**.*

```
y = 3   ->   (0, 0, 0, 1, 0, 0, 0, 0, 0, 0)      자리 번호는 0 부터 / positions start at 0
```

**벡터 이어 붙이기(concatenation).** 이미지 x 는 784개의 숫자, 원-핫 y 는 10개의 숫자입니다. 두 벡터를 옆으로 이어 붙이면 794개의 숫자가 되고, 인코더는 이것을 입력으로 받습니다. 디코더도 같은 방법으로 z 와 y 를 이어 붙여 받습니다.
*Concatenation. The image x is 784 numbers and the one-hot y is 10 numbers. Placing them side by side gives 794 numbers, which the encoder takes as input. The decoder takes z and y joined the same way.*

| 함수 / Function | 뜻 / Meaning | 예 / Example |
|---|---|---|
| `F.one_hot(y, 10).float()` | 정수 라벨을 길이 10 의 원-핫 벡터로<br>Integer label to a length-10 one-hot vector | `F.one_hot(torch.tensor([2]), 10)` → `[[0,0,1,0,0,0,0,0,0,0]]` |
| `torch.cat([a, b], dim=1)` | 두 벡터(배치의 각 행)를 옆으로 이어 붙이기<br>Join two vectors side by side (row by row in a batch) | 크기 (64, 5) 와 (64, 3) → (64, 8) / shapes (64, 5) and (64, 3) → (64, 8) |

아래 셀의 인코더는 이미 `torch.cat([x, y1h], dim=1)` 로 x 와 y 를 받습니다. **빈칸은 디코더가 무엇을 받아야 하는가** 한 줄입니다.
*The encoder below already takes x and y through `torch.cat([x, y1h], dim=1)`. The blank is one line: what must the decoder receive?*

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   x      : 이미지 배치를 784개 숫자로 편 것. 크기 (B, 784) / batch of images flattened to 784 numbers, shape (B, 784)
#   y1h    : 라벨의 원-핫 벡터. 크기 (B, 10) / one-hot labels, shape (B, 10)
#   z      : 잠재 코드. 크기 (B, 2). 4주차처럼 2차원 / latent code, shape (B, 2), two dimensions as in Week 4
#   h      : 디코더 첫 층에 들어갈 입력   <- 여기가 빈칸 / input to the decoder's first layer   <- this is the blank
#   mu, logvar : 인코더가 내는 q(z | x, y) 의 평균과 로그분산 / mean and log-variance of q(z | x, y)
#
# 참고 / Reference   torch.cat([a, b], dim=1)
#   배치의 각 행에서 a 와 b 를 옆으로 이어 붙인다.
#   Joins a and b side by side, row by row.
#   예 / e.g.   torch.cat([torch.zeros(4, 5), torch.ones(4, 3)], dim=1).shape  ->  (4, 8)

class CVAE(nn.Module):
    def __init__(self, z_dim=2, n_classes=10, hidden=256):
        super().__init__()
        self.n_classes = n_classes
        self.enc = nn.Sequential(nn.Linear(784 + n_classes, hidden), nn.ReLU())   # 794 -> 256
        self.enc_mu = nn.Linear(hidden, z_dim)                                    # 평균 / mean
        self.enc_logvar = nn.Linear(hidden, z_dim)                                # 로그분산 / log-variance
        self.dec = nn.Sequential(nn.Linear(z_dim + n_classes, hidden), nn.ReLU(), # 12 -> 256
                                 nn.Linear(hidden, 784))                          # 256 -> 784 로짓 / logits

    def encode(self, x, y1h):
        h = self.enc(torch.cat([x, y1h], dim=1))   # 인코더는 x 와 y 를 함께 본다 / the encoder sees x and y together
        return self.enc_mu(h), self.enc_logvar(h)

    def decode(self, z, y1h):
        # TODO. 디코더가 "어떤 숫자를 그릴지"를 알려면 무엇을 받아야 하나요?
        #       z 와 무엇을 이어 붙여 h 에 넣으세요. 한 줄입니다.
        # TODO. To know which digit to draw, what must the decoder receive besides z?
        #       Join z with it and put the result in h. One line.
        h = None

        if h is None:
            raise NotImplementedError(BLANK_MSG)

        return self.dec(h)                         # 픽셀마다 로짓 784개 / 784 logits, one per pixel

model_check = CVAE()
z_try = torch.zeros(4, 2)                                  # z 네 개 / four codes
y_try = F.one_hot(torch.tensor([0, 3, 7, 9]), 10).float()  # 라벨 네 개 / four labels
print("디코더 출력 크기 / decoder output shape:", tuple(model_check.decode(z_try, y_try).shape))
# 기대값 / Expected   (4, 784)
#   "mat1 and mat2 shapes cannot be multiplied (4x2 and 12x256)" 가 나오면 y 를 이어 붙이지 않은 것입니다.
#   "mat1 and mat2 shapes cannot be multiplied (4x2 and 12x256)" means y was not joined to z.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

class CVAE(nn.Module):
    def __init__(self, z_dim=2, n_classes=10, hidden=256):
        super().__init__()
        self.n_classes = n_classes
        self.enc = nn.Sequential(nn.Linear(784 + n_classes, hidden), nn.ReLU())   # 794 -> 256
        self.enc_mu = nn.Linear(hidden, z_dim)                                    # 평균 / mean
        self.enc_logvar = nn.Linear(hidden, z_dim)                                # 로그분산 / log-variance
        self.dec = nn.Sequential(nn.Linear(z_dim + n_classes, hidden), nn.ReLU(), # 12 -> 256
                                 nn.Linear(hidden, 784))                          # 256 -> 784 로짓 / logits

    def encode(self, x, y1h):
        h = self.enc(torch.cat([x, y1h], dim=1))   # 인코더는 x 와 y 를 함께 본다 / the encoder sees x and y together
        return self.enc_mu(h), self.enc_logvar(h)

    def decode(self, z, y1h):
        h = torch.cat([z, y1h], dim=1)             # 빈칸이었던 줄. 디코더도 라벨을 본다
                                                   # the blank. The decoder sees the label too

        if h is None:
            raise NotImplementedError(BLANK_MSG)

        return self.dec(h)                         # 픽셀마다 로짓 784개 / 784 logits, one per pixel

model_check = CVAE()
z_try = torch.zeros(4, 2)                                  # z 네 개 / four codes
y_try = F.one_hot(torch.tensor([0, 3, 7, 9]), 10).float()  # 라벨 네 개 / four labels
print("디코더 출력 크기 / decoder output shape:", tuple(model_check.decode(z_try, y_try).shape))

## §3 CVAE 학습 / Train the CVAE

손실은 4주차와 **같은 두 항**입니다. 바뀐 것은 인코더와 디코더가 y 도 받는다는 것뿐이고, 사전분포는 4주차와 같은 N(0, I) 로 두며 y 를 보지 않습니다(슬라이드 「Both Networks See y. The Prior Stays Fixed」).
*The loss has the **same two terms** as Week 4. The only change is that the encoder and the decoder also receive y; the prior stays N(0, I), as in Week 4, and does not see y.*

$$\text{loss}(x, y) \;=\; -\,\mathbb{E}_{q(z \mid x, y)}\big[\log p(x \mid z, y)\big] \;+\; \mathrm{KL}\big(q(z \mid x, y)\,\|\,p(z)\big)$$

코드에서는 z 를 한 번 뽑아 첫 항의 평균을 어림합니다(4주차와 같음). *In code, one draw of z estimates the average in the first term, as in Week 4.*

- 첫 항은 4주차처럼 베르누이 디코더의 이진 교차엔트로피(BCE) 입니다. *The first term is the Bernoulli decoder's binary cross-entropy, as in Week 4.*
- 둘째 항은 4주차에 유도한 가우시안 KL 닫힌 형 ½ Σ ( μ² + σ² − 1 − ln σ² ) 입니다. *The second term is the closed-form Gaussian KL derived in Week 4.*

`reparameterize` 와 `kl_term` 은 4주차 노트북에서 여러분이 직접 쓰는 두 함수와 같은 일을 합니다. 이름이나 인자 모양은 조금 다를 수 있습니다. 여기서는 **다시 쓰지 않고 완성된 것을 줍니다.**
*`reparameterize` and `kl_term` do the same jobs as the two functions you write in the Week 4 notebook; names or arguments may differ slightly. Here they are given, not rewritten.*

In [ ]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)      # logvar = ln σ² 이므로 σ = exp(½ logvar) / since logvar = ln σ², σ = exp(½ logvar)
    eps = torch.randn_like(std)        # ε ~ N(0, 1), 학습하지 않는 잡음 / fixed noise, nothing learnable
    return mu + std * eps              # z = μ + σ · ε  (4주차 덱 「Draw the Noise First, Then Calculate the Code」)

def kl_term(mu, logvar):
    # 이미지마다 잠재 좌표에 대해 더한 KL(q(z | x, y) ‖ N(0, I)) / per-image KL, summed over latent coordinates
    return 0.5 * torch.sum(mu**2 + logvar.exp() - 1 - logvar, dim=1)

def cvae_loss(model, x, y):
    y1h = F.one_hot(y, 10).float()                                   # 라벨을 원-핫으로 / label to one-hot
    mu, logvar = model.encode(x, y1h)                                # q(z | x, y)
    z = reparameterize(mu, logvar)                                   # z 한 개 뽑기 / one draw of z
    logits = model.decode(z, y1h)                                    # p(x | z, y) 의 로짓 / logits of p(x | z, y)
    recon = F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=1)   # 이미지당 BCE / BCE per image
    kl = kl_term(mu, logvar)
    return (recon + kl).mean(), recon.mean().item(), kl.mean().item()

torch.manual_seed(0)
cvae = CVAE().to(device)
opt = torch.optim.Adam(cvae.parameters(), lr=1e-3)
loader = DataLoader(train_set, batch_size=128, shuffle=True)

for epoch in range(1, 6):              # 다섯 번 훑기 / five passes over the data
    tot, rec, kls, n = 0.0, 0.0, 0.0, 0
    for xb, yb in loader:
        xb = xb.view(-1, 784).to(device); yb = yb.to(device)
        loss, r, k = cvae_loss(cvae, xb, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); rec += r; kls += k; n += 1
    print(f"epoch {epoch}   손실 / loss {tot/n:6.1f}   복원 / recon {rec/n:6.1f}   KL {kls/n:4.2f}")
# 기대값 / Expected   마지막 줄이 손실 약 139.9, 복원 약 135.8, KL 약 4.15
#   The last line should read about loss 139.9, recon 135.8, KL 4.15.
#   CPU 와 GPU, 라이브러리 버전에 따라 소수점 아래가 조금 다를 수 있습니다.
#   CPU vs GPU and library versions can change the decimals slightly.
#   KL 이 0.1 아래로 붙어 있으면 4주차의 posterior collapse 신호입니다. kl_term 부호를 확인하세요.
#   A KL stuck below 0.1 is the Week 4 posterior-collapse signal. Check the sign in kl_term.

## §4 숫자를 골라 생성하기 / Generate a digit you choose

생성할 때는 인코더를 쓰지 않습니다. **y 를 고르고**, z 를 사전분포 N(0, I) 에서 뽑고, 디코더에 둘을 함께 넣습니다.
*Generation does not use the encoder. **Choose y**, draw z from the prior N(0, I), and give both to the decoder.*

아래 그림은 슬라이드 「Choose the Label, Then Draw」와 같은 배치입니다. **한 행 안에서 z 가 같고**, 한 열 안에서 y 가 같습니다. 여섯 개의 z 를 뽑아 행마다 하나씩 고정했습니다.
*The figure below has the same layout as the slide. **Each row shares one z** and each column shares one y. Six codes z were drawn and each row keeps one of them.*

In [ ]:
torch.manual_seed(1)
z_rows = torch.randn(6, 2).to(device)            # 사전분포에서 z 여섯 개 / six codes from the prior
fig, axes = plt.subplots(6, 10, figsize=(7, 4.4))
cvae.eval()
with torch.no_grad():
    for digit in range(10):
        y1h = F.one_hot(torch.full((6,), digit), 10).float().to(device)   # 같은 라벨 여섯 개 / the same label six times
        imgs = torch.sigmoid(cvae.decode(z_rows, y1h)).view(-1, 28, 28).cpu()   # 로짓을 확률로 / logits to probabilities
        for j in range(6):
            axes[j, digit].imshow(imgs[j], cmap="gray"); axes[j, digit].axis("off")
        axes[0, digit].set_title(f"y={digit}", fontsize=7)
for j in range(6):
    axes[j, 0].text(-0.1, 0.5, f"z{j + 1}", transform=axes[j, 0].transAxes, fontsize=7, ha="right", va="center")
plt.suptitle("rows: the same z    columns: chosen label y", fontsize=8)   # 그림 글자는 영어만. Colab 기본 글꼴에 한글이 없다 / English only: Colab's default font has no Hangul
plt.tight_layout(); plt.show()
# 보는 법 / How to read it (슬라이드 「Choose the Label, Then Draw」와 같은 배치 / same layout as the slide)
#   열을 따라 내려가면 모두 같은 숫자여야 합니다. 모형은 라벨이 요청한 숫자를 만들도록 학습됩니다.
#   Down a column every picture should be the same digit. The model learns to make the digit the label asks for.
#   행을 따라가면 굵기나 기울기 같은 "쓰는 방식"이 비슷해 보이는 경향이 있습니다. z 가 그것을 담는 경향이 있습니다.
#   Along a row, the "way of writing" such as thickness or slant tends to look alike. z tends to carry that.
#   열마다 다른 숫자가 섞여 나오면 §2 의 디코더가 y 를 받지 못하고 있는 것입니다.
#   If a column mixes different digits, the decoder in §2 is not receiving y.

**해석 / Interpretation.** 라벨이 숫자의 정체를 맡으니, z 는 라벨이 알려 주지 않는 나머지(쓰는 방식)를 담게 되는 경향이 있습니다. "항상 그렇다"는 보장은 아닙니다. 모형이 그렇게 강제하지 않기 때문입니다.
*The label takes care of which digit, so z tends to carry what the label does not say (the way of writing). This is a tendency, not a guarantee; nothing in the model forces it.*

**CVAE 는 흐림(blur)을 고치지 않습니다.** 위 그림도 4주차처럼 흐릿합니다. CVAE 가 더한 것은 "무엇을 생성할지 고르는 손잡이"입니다.
*A CVAE does not fix blur. The pictures above are as soft as in Week 4. What a CVAE adds is a handle for choosing what to generate.*

## §5 가장 가까운 코드 고르기, 손으로 먼저 / The nearest code, by hand first

슬라이드 「From Encoder Output to Quantized Latent」와 같은 숫자입니다. 코드북에 2차원 코드 세 개가 있습니다.
*The same numbers as the slide. A codebook holds three 2-dimensional codes.*

```
e1 = (0, 0)   e2 = (3, 4)   e3 = (5, 1)        인코더 출력 / encoder output  z_e = (4, 4)

제곱거리 / squared distance  ‖z_e − e‖² = (차이₁)² + (차이₂)²  / (difference 1)² + (difference 2)²
  e1 :  (4 − 0)² + (4 − 0)² = 16 + 16 = 32
  e2 :  (4 − 3)² + (4 − 4)² =  1 +  0 =  1     <- 가장 작다 / smallest
  e3 :  (4 − 5)² + (4 − 1)² =  1 +  9 = 10
```

그래서 z_e 는 e2 로 바뀝니다. **코드에서는 번호가 0 부터 시작**하므로 e1, e2, e3 이 0, 1, 2 번이고, 답은 **1번**입니다.
*So z_e is replaced by e2. **In code, numbering starts at 0**, so e1, e2, e3 are indices 0, 1, 2, and the answer is **index 1**.*

`argmin` 은 "가장 작은 값이 **몇 번째**에 있는가"를 돌려줍니다. 가장 작은 값 그 자체가 아닙니다.
*`argmin` returns **where** the smallest value is, not the smallest value itself.*

이 예에서는 가장 작은 거리도 1, 답의 번호도 1 이라 헷갈리기 쉽습니다. 앞의 1 은 거리(값), 뒤의 1 은 번호(자리)입니다.
*In this example the smallest distance is 1 and the answer is also index 1. The first 1 is a distance (a value), the second is an index (a position).*

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   z_e       : 인코더 출력 벡터들. 크기 (N, D) / encoder output vectors, shape (N, D)
#   codebook  : 코드 K 개. 크기 (K, D) / the K codes, shape (K, D)
#   dists     : 각 z_e 와 각 코드 사이 제곱거리. 크기 (N, K) / squared distance from each z_e to each code, shape (N, K)
#   k         : 각 z_e 에 가장 가까운 코드의 번호. 크기 (N,)   <- 여기가 빈칸
#               index of the nearest code for each z_e, shape (N,)   <- this is the blank
#
# 참고 / Reference   t.argmin(dim=1)
#   각 행에서 가장 작은 값이 있는 열 번호를 돌려준다.
#   Returns, for each row, the column index of the smallest value.
#   예 / e.g.   torch.tensor([[5., 2., 9.], [1., 7., 3.]]).argmin(dim=1)  ->  tensor([1, 0])

def nearest_code(z_e, codebook):
    dists = ((z_e[:, None, :] - codebook[None, :, :]) ** 2).sum(dim=2)   # (N, K) 제곱거리 / squared distances

    # TODO. 각 행(각 z_e)에서 거리가 가장 작은 코드의 번호를 k 에 넣으세요. 한 줄입니다.
    # TODO. Put into k the index of the code with the smallest distance, for each row. One line.
    k = None

    if k is None:
        raise NotImplementedError(BLANK_MSG)

    return k, dists

codebook_toy = torch.tensor([[0., 0.], [3., 4.], [5., 1.]])   # e1, e2, e3
z_e_toy = torch.tensor([[4., 4.]])
k_toy, d_toy = nearest_code(z_e_toy, codebook_toy)
print("거리 / distances:", d_toy.tolist(), "   고른 번호 / chosen index:", k_toy.tolist())
# 기대값 / Expected   거리 [[32.0, 1.0, 10.0]], 번호 [1]  / distances [[32.0, 1.0, 10.0]], index [1]
#   [0] 이 나오면 argmin 대신 argmax 를 쓴 것입니다. 가장 먼 코드(거리 32)를 고른 셈입니다.
#   [0] means argmax was used instead of argmin, which picks the farthest code (distance 32).
#   [1.0] 처럼 소수점이 붙으면 번호가 아니라 가장 작은 거리 값(1.0)을 넣은 것입니다. argmin 을 쓰세요.
#   [1.0], with a decimal point, means the smallest distance itself (1.0) went in, not its index. Use argmin.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

def nearest_code(z_e, codebook):
    dists = ((z_e[:, None, :] - codebook[None, :, :]) ** 2).sum(dim=2)   # (N, K) 제곱거리 / squared distances

    k = dists.argmin(dim=1)      # 빈칸이었던 줄. 행마다 가장 작은 거리의 번호
                                 # the blank. Index of the smallest distance in each row

    if k is None:
        raise NotImplementedError(BLANK_MSG)

    return k, dists

codebook_toy = torch.tensor([[0., 0.], [3., 4.], [5., 1.]])   # e1, e2, e3
z_e_toy = torch.tensor([[4., 4.]])
k_toy, d_toy = nearest_code(z_e_toy, codebook_toy)
print("거리 / distances:", d_toy.tolist(), "   고른 번호 / chosen index:", k_toy.tolist())

## §6 고르기는 기울기를 막는다, 한 줄로 통과시키기 / A choice blocks the gradient; one line lets it through

z_e 를 조금 움직여도 가장 가까운 코드는 대개 그대로입니다. 그래서 z_q(고른 코드)는 z_e 에 대해 **계단처럼 평평**하고, 기울기는 0 입니다. 인코더는 복원 손실에서 배울 신호를 받지 못합니다. 4주차에도 인코더까지 기울기가 오지 않았지만 원인과 해법이 다릅니다. 4주차는 뽑기를 지나는 경로가 없어 재매개변수화로 진짜 경로를 만들었고, 여기는 기울기가 0 이라 straight-through 로 대신 쓸 기울기를 줍니다.
*Moving z_e a little usually leaves the nearest code unchanged. So z_q (the chosen code) is **flat like a staircase** in z_e and its gradient is 0. The encoder gets no signal from the reconstruction loss. Week 4 also had no gradient reaching the encoder, but the cause and the fix differ: there the draw left no path and reparameterization built a real one; here the slope is 0 and straight-through supplies a substitute gradient.*

**straight-through.** 앞으로 계산할 때는 z_q 를 쓰고, 뒤로 기울기를 보낼 때는 z_q 가 z_e 였던 것처럼 기울기를 **그대로 복사**합니다. 이것은 고르기의 정확한 기울기(0)가 아니라 **대신 쓰는 기울기(surrogate gradient)** 입니다.
*Straight-through. Use z_q going forward; going backward, **copy** the gradient as if z_q were z_e. This is a **surrogate gradient**, not the exact gradient of the choice (which is 0).*

`t.detach()` 는 "t 의 값은 쓰되, 기울기 계산에서는 상수로 본다"는 뜻입니다.
*`t.detach()` means "use the value of t, but treat it as a constant when computing gradients".*

```
z_e + (z_q − z_e).detach()
  값   / value     : z_e + z_q − z_e = z_q                 앞으로는 고른 코드 / forward: the chosen code
  기울기 / gradient : d/dz_e [ z_e + 상수 / constant ] = 1   뒤로는 그대로 복사 / backward: copied straight through
```

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   z_e     : 인코더 출력 / encoder output
#   z_q     : 가장 가까운 코드. 값만 쓰고 기울기는 z_e 로 이어지지 않는다
#             the nearest code; its value is used but no gradient links it to z_e
#   z_q_st  : 앞으로는 z_q 의 값, 뒤로는 z_e 로 기울기를 복사하는 텐서   <- 여기가 빈칸
#             a tensor with the value of z_q going forward and a gradient copied to z_e going backward   <- the blank
#
# 참고 / Reference   t.detach()
#   t 와 같은 값을 갖되, 기울기 계산에서는 상수로 취급되는 텐서를 돌려준다.
#   Returns a tensor with the same value as t that is treated as a constant for gradients.
#   예 / e.g.   a = torch.tensor(2., requires_grad=True);  (a * a.detach()).backward();  a.grad  ->  2.0

def straight_through(z_e, z_q):
    # TODO. 위 상자의 식을 그대로 코드로 옮겨 z_q_st 에 넣으세요. 한 줄입니다.
    # TODO. Write the expression in the box above as code and put it in z_q_st. One line.
    z_q_st = None

    if z_q_st is None:
        raise NotImplementedError(BLANK_MSG)

    return z_q_st

z_e_chk = torch.tensor([[4., 4.]], requires_grad=True)
z_q_chk = codebook_toy[nearest_code(z_e_chk.detach(), codebook_toy)[0]]    # 고른 코드 (3, 4) / the chosen code (3, 4)
out = straight_through(z_e_chk, z_q_chk)
out.sum().backward()
print("앞으로 가는 값 / forward value:", out.tolist(), "   z_e 에 닿은 기울기 / gradient at z_e:", z_e_chk.grad.tolist())
# 기대값 / Expected   값 [[3.0, 4.0]], 기울기 [[1.0, 1.0]] / value [[3.0, 4.0]], gradient [[1.0, 1.0]]
#   값이 [[4.0, 4.0]] 이면 z_e 를 그대로 돌려준 것입니다. 앞으로는 z_q 가 가야 합니다.
#   A value of [[4.0, 4.0]] means z_e was returned. Going forward it must be z_q.
#   "does not require grad" 오류는 z_e 까지 detach 한 것입니다. detach 는 괄호 (z_q − z_e) 에만 붙입니다.
#   A "does not require grad" error means z_e was detached too. Detach only the bracket (z_q − z_e).

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

def straight_through(z_e, z_q):
    z_q_st = z_e + (z_q - z_e).detach()   # 빈칸이었던 줄. 값은 z_q, 기울기는 z_e 로 그대로
                                          # the blank. Value of z_q, gradient straight to z_e

    if z_q_st is None:
        raise NotImplementedError(BLANK_MSG)

    return z_q_st

z_e_chk = torch.tensor([[4., 4.]], requires_grad=True)
z_q_chk = codebook_toy[nearest_code(z_e_chk.detach(), codebook_toy)[0]]    # 고른 코드 (3, 4) / the chosen code (3, 4)
out = straight_through(z_e_chk, z_q_chk)
out.sum().backward()
print("앞으로 가는 값 / forward value:", out.tolist(), "   z_e 에 닿은 기울기 / gradient at z_e:", z_e_chk.grad.tolist())

## §7 VQ-VAE 학습 / Train a VQ-VAE

인코더는 28 x 28 이미지를 **7 x 7 격자**로 줄이고, 격자의 칸마다 D = 16 차원 벡터 z_e 를 냅니다. 칸마다 따로 가장 가까운 코드로 바꾸므로, 이미지 한 장은 **0 ~ 31 사이 번호 49개**로 표현됩니다.
*The encoder shrinks a 28 x 28 image to a **7 x 7 grid** and outputs a D = 16 vector z_e for every cell. Each cell is snapped to its own nearest code, so one image becomes **49 indices between 0 and 31**.*

손실은 세 항입니다(슬라이드 「The Full VQ-VAE Loss」). *The loss has three terms.*

| 항 / Term | 식 / Formula | 무엇을 학습시키나 / What it trains |
|---|---|---|
| 복원 / reconstruction | BCE(x, decoder(z_q_st)) | 디코더, 그리고 straight-through 로 인코더<br>the decoder, and the encoder through straight-through |
| 코드북 / codebook | ‖ sg[z_e] − e_k ‖² | 고른 코드. 인코더 출력 쪽으로 당긴다<br>the chosen code, pulled toward the encoder output |
| 커밋먼트 / commitment | β_VQ ‖ z_e − sg[e_k] ‖², β_VQ = 0.25 | 인코더. 출력을 고른 코드 쪽으로 당긴다<br>the encoder, its output pulled toward the chosen code |

sg[·] 는 stop-gradient, 코드에서는 `.detach()` 입니다. β_VQ 는 커밋먼트 가중치이고, 4주차의 KL 가중치 β 와 **다른 양**입니다(슬라이드 「How Does the Codebook Learn?」 · 「The Commitment Term Moves the Encoder Instead」).
*sg[·] is stop-gradient, `.detach()` in code. β_VQ is the commitment weight, a **different quantity** from the Week 4 KL weight β.*

KL 항이 없는 이유: 인코더가 코드 하나를 확률 1 로 고르고 사전분포를 K 개 코드에 균등하게 두면, KL 이 늘 ln K 인 상수가 되어 학습에 영향을 주지 않습니다(슬라이드 「Why Is There No KL Loss in VQ-VAE?」).
*Why no KL term: the encoder picks one code with probability 1 and the prior is uniform over K codes, so the KL is always the constant ln K and does not affect training.*

In [ ]:
K, D, BETA_VQ = 32, 16, 0.25          # 코드 수, 코드 차원, 커밋먼트 가중치 β_VQ / number of codes, code dimension, commitment weight β_VQ

class VQVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(),     # 28 -> 14
                                 nn.Conv2d(32, 32, 4, 2, 1), nn.ReLU(),    # 14 -> 7
                                 nn.Conv2d(32, D, 1))                      # 칸마다 D 차원 z_e / a D-vector per cell
        self.codebook = nn.Parameter(torch.empty(K, D).uniform_(-1 / K, 1 / K))   # 학습되는 코드 K 개 / K learnable codes
        self.dec = nn.Sequential(nn.Conv2d(D, 32, 3, 1, 1), nn.ReLU(),
                                 nn.ConvTranspose2d(32, 32, 4, 2, 1), nn.ReLU(),   # 7 -> 14
                                 nn.ConvTranspose2d(32, 1, 4, 2, 1))               # 14 -> 28 로짓 / logits

    def quantize(self, z_e):
        B, _, H, W = z_e.shape
        flat = z_e.permute(0, 2, 3, 1).reshape(-1, D)          # 칸마다 한 행 / one row per grid cell
        k, _ = nearest_code(flat.detach(), self.codebook.detach())   # §5 의 함수 / the §5 function
        z_q = self.codebook[k].view(B, H, W, D).permute(0, 3, 1, 2)  # 고른 코드로 바꾼 격자 / grid of chosen codes
        return z_q, k.view(B, H, W)

    def forward(self, x):
        z_e = self.enc(x)
        z_q, k = self.quantize(z_e)
        logits = self.dec(straight_through(z_e, z_q))          # §6 의 함수 / the §6 function
        return logits, z_e, z_q, k

def vq_loss(model, x):
    logits, z_e, z_q, k = model(x)
    recon = F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=(1, 2, 3))   # 이미지당 / per image
    codebook_term = ((z_e.detach() - z_q) ** 2).sum(dim=(1, 2, 3))          # ‖ sg[z_e] − e_k ‖²
    commit_term = BETA_VQ * ((z_e - z_q.detach()) ** 2).sum(dim=(1, 2, 3))  # β_VQ ‖ z_e − sg[e_k] ‖²
    return (recon + codebook_term + commit_term).mean(), recon.mean().item(), k

torch.manual_seed(0)
vq = VQVAE().to(device)
# 코드북을 인코더 출력 근처에서 시작한다. 인코더 출력 256장 x 49칸 중 K 개를 무작위로 골라 첫 코드로 쓴다.
# Start the codebook near the encoder outputs: K random vectors out of 256 images x 49 cells become the first codes.
# 이 줄이 없으면 처음부터 멀리 있는 코드는 한 번도 뽑히지 않아 신호를 받지 못한다(§9 의 죽은 코드).
# Without it, codes that start far away are never chosen and never get a signal (the dead codes of §9).
with torch.no_grad():
    x_init = torch.stack([train_set[i][0] for i in range(256)]).to(device)      # 이미지 256장 / 256 images
    z_init = vq.enc(x_init).permute(0, 2, 3, 1).reshape(-1, D)                 # 칸마다 한 행 / one row per cell
    vq.codebook.data.copy_(z_init[torch.randperm(len(z_init))[:K]])            # K 개를 첫 코드로 / K of them as the first codes
opt_vq = torch.optim.Adam(vq.parameters(), lr=2e-3)
vq_loader = DataLoader(Subset(train_set, range(20000)), batch_size=128, shuffle=True)   # 시간을 줄이려 2만 장 / 20,000 images to save time

for epoch in range(1, 4):              # 세 번 훑기 / three passes
    rec, n = 0.0, 0
    used = torch.zeros(K)
    for xb, _ in vq_loader:
        xb = xb.to(device)
        loss, r, k = vq_loss(vq, xb)
        opt_vq.zero_grad(); loss.backward(); opt_vq.step()
        rec += r; n += 1
        used += torch.bincount(k.flatten().cpu(), minlength=K)
    print(f"epoch {epoch}   복원 / recon {rec/n:6.1f}   이번 에폭에 쓰인 코드 / codes used this epoch {int((used > 0).sum())} / {K}")
# 기대값 / Expected   마지막 줄이 복원 약 79, 쓰인 코드 약 25 / 32
#   The last line should read about recon 79, codes used 25 of 32.
#   CPU 와 GPU 에 따라 조금 다를 수 있습니다. 시드를 바꿔 본 세 번의 실행에서 복원 71~79, 코드 25~30 이었습니다.
#   CPU vs GPU can differ slightly. Three runs with other seeds gave recon 71 to 79 and 25 to 30 codes.
#   §6 의 검사를 통과했다면 이 셀에서 따로 고칠 것은 없습니다.
#   If the §6 check passed, there is nothing to fix in this cell.

## §8 무엇을 얻었나 / What did we get?

아래 첫 두 줄은 시험 이미지와 그 복원, 셋째 줄은 **49개 번호를 서로 독립으로, 균등하게 골라** 디코딩한 것입니다. 학습 때 쓴 균등 사전분포에서 뽑은 표본입니다. 슬라이드 「Why Do Random Code Grids Not Look Like Digits?」와 같은 그림입니다.
*The first two rows are test images and their reconstructions. The third row decodes **49 indices drawn independently and uniformly**, that is, a sample from the uniform prior used in training. The same figure as the slide.*

In [ ]:
vq.eval()
x_test = torch.stack([test_set[i][0] for i in range(8)]).to(device)
with torch.no_grad():
    logits, _, _, k_test = vq(x_test)
    recon_img = torch.sigmoid(logits).cpu()
    k_rand = torch.randint(0, K, (8, 7, 7)).to(device)                        # 번호를 균등하게 무작위로 / indices uniformly at random
    z_rand = vq.codebook[k_rand].permute(0, 3, 1, 2)                          # 번호를 코드 벡터로 / indices to code vectors
    rand_img = torch.sigmoid(vq.dec(z_rand)).cpu()

fig, axes = plt.subplots(3, 8, figsize=(8, 3.2))
rows = [(x_test.cpu(), "test images"), (recon_img, "reconstructions"), (rand_img, "decoded random indices")]   # 그림 글자는 영어만 / English only in figures
for r, (imgs, name) in enumerate(rows):
    for j in range(8):
        axes[r, j].imshow(imgs[j, 0], cmap="gray"); axes[r, j].axis("off")
    axes[r, 0].set_title(name, fontsize=7, loc="left")
plt.tight_layout(); plt.show()
print("첫 시험 이미지의 7 x 7 번호 격자 / the 7 x 7 index grid of the first test image")
print(k_test[0].cpu().numpy())
# 보는 법 / How to read it
#   복원은 4주차 VAE 보다 또렷한 편입니다. 번호 49개가 이미지 한 장을 거의 그대로 담습니다.
#   Reconstructions tend to be sharper than the Week 4 VAE. Forty-nine indices hold most of an image.
#   셋째 줄은 숫자처럼 보이지 않습니다. 독립 균등 사전분포는 이웃한 칸의 번호가 서로 어울리는지를 반영하지 못합니다.
#   The third row does not look like digits. An independent uniform prior ignores which neighbouring indices go together.
#   그 "어울림"을 배우는 두 번째 모형(번호 격자 위의 사전분포)이 있어야 VQ-VAE 로 새 이미지를 만들 수 있습니다.
#   A second model over index grids (a prior over codes) is what lets a VQ-VAE generate new images.

## §9 예측 다시 읽기 / Re-read your predictions

1. **같은 z, 다른 y.** 숫자가 바뀌고, 쓰는 방식은 비슷하게 남는 경향이 있었습니다. 이 실행에서는 라벨이 숫자를 정하고, z 는 쓰는 방식을 담는 경향이 있었습니다.
   *Same z, different y. The digit changed; the way of writing tended to stay. In this run the label set the digit, and z tended to carry the way of writing.*
2. **인코더에만 라벨.** 고를 수 없습니다. 생성할 때는 인코더를 쓰지 않으므로 디코더가 y 를 못 보면 y 는 아무 일도 하지 않습니다.
   *Label only in the encoder. You cannot choose. Generation skips the encoder, so if the decoder never sees y, y does nothing.*
3. **K = 32.** 한 칸은 32가지 중 하나만 가질 수 있습니다. 학습 뒤에도 32개가 다 쓰이지는 않았습니다(§7 에서 약 25개). 한 번도 고르지 않은 코드는 코드북 항의 신호를 받지 못해 그대로 남습니다. 이것을 **죽은 코드(dead codes)** 라고 부릅니다.
   *K = 32. A cell can take only one of 32 vectors. After training, not all 32 were used (about 25 in §7). A code that is never chosen gets no signal from the codebook term and stays where it started. These are called dead codes.*

**흔한 실수 세 가지 / Three common mistakes**
- 라벨을 인코더에만 넣기 / giving the label only to the encoder
- `detach` 를 z_e 에 붙이거나 아예 빼먹기 / detaching z_e, or forgetting detach
- 독립 균등 사전분포에서 뽑은 번호 격자가 데이터처럼 보이기를 기대하기 / expecting indices drawn independently and uniformly to look like data

### 더 해 보고 싶다면 / For the curious (Optional)
§7 셀에서 `with torch.no_grad():` 로 시작하는 코드북 초기화 네 줄을 지우고 다시 학습해 보세요. 이 노트북을 만든 환경에서는 쓰인 코드가 **32개 중 11개**로 줄었습니다. 처음부터 인코더 출력에서 멀리 놓인 코드는 한 번도 뽑히지 않고, 뽑히지 않은 코드는 코드북 항의 신호를 받지 못하기 때문입니다.
*Delete the four codebook-initialization lines starting with `with torch.no_grad():` in §7 and train again. In the environment used to build this notebook, only **11 of 32** codes were used. Codes that start far from the encoder outputs are never chosen, and a code that is never chosen gets no signal from the codebook term.*

### 더 읽고 돌려 볼 코드 / More code to read and run (Optional)
- 교재 저자 저장소 `jmtomczak/intro_dgm` (MIT 라이선스)의 `vaes/vae_example.ipynb` 는 8 x 8 숫자 데이터(scikit-learn Digits)를 써서 노트북 PC 의 CPU 에서도 몇 분 안에 돕니다. 4주차 VAE 를 다른 사람의 코드로 한 번 더 읽어 보기 좋습니다.
  *The textbook author's repository `jmtomczak/intro_dgm` (MIT licence) has `vaes/vae_example.ipynb`, which uses 8 x 8 digits (scikit-learn Digits) and runs on a laptop CPU in minutes. A good second reading of the Week 4 VAE.*
- 같은 저장소의 `vaes/vae_priors_example.ipynb` 는 반올림에 straight-through 를 씁니다. §6 의 `detach` 한 줄 대신, 뒤로 갈 때 기울기를 그대로 돌려주는 함수를 직접 정의하는 방식입니다. 생각은 같습니다.
  *`vaes/vae_priors_example.ipynb` in the same repository uses straight-through for rounding. Instead of the §6 `detach` line it defines a function whose backward pass returns the gradient unchanged. Same idea.*